# YOLOv11s 공구 검출 재학습 — top_view_v2

**데이터셋**: 단일 객체 614장 + mix 200장 (검수 완료) = 801장  
**Split**: Train 561 / Valid 160 / Test 80  
**모델**: YOLOv11s  
**목표**: mAP@0.5 ≥ 0.85, ratchet_wrench / spanner_16mm 오인식 개선

---
**실행 전 확인**
- 런타임 → 런타임 유형 변경 → **GPU** (T4 이상)
- 구글 드라이브 마운트 (학습 결과 저장용)

In [ ]:
# GPU 확인
!nvidia-smi

In [ ]:
# 패키지 설치
!pip install -q ultralytics roboflow

In [ ]:
# 구글 드라이브 마운트 (결과 저장용)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Roboflow에서 데이터셋 다운로드
from roboflow import Roboflow

rf = Roboflow(api_key="TRSBojryEBlxSSsZyv3f")
project = rf.workspace("yeonseop9999-gmail-com").project("final-project-kir4p")
version = project.version(1)
dataset = version.download("yolov11")

print("데이터셋 경로:", dataset.location)

In [ ]:
# 데이터셋 구성 확인
import os

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset.location, split, "images")
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f"{split}: {count}장")

# data.yaml 확인
!cat {dataset.location}/data.yaml

In [ ]:
# 학습 실행
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data=dataset.location + "/data.yaml",
    epochs=200,
    imgsz=640,
    batch=16,
    device="cuda",
    project="/content/drive/MyDrive/Final_project/runs",
    name="top_view_v2",
    patience=50,
    save=True,
    plots=True,
    # 탑뷰 augmentation
    degrees=180.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.5,
    perspective=0.0,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    erasing=0.3,
)

In [ ]:
# 학습 결과 확인
metrics = results.results_dict
map50 = metrics.get("metrics/mAP50(B)", 0.0)
map50_95 = metrics.get("metrics/mAP50-95(B)", 0.0)

print(f"mAP@0.5     : {map50:.3f}  (수락 기준 ≥ 0.85)")
print(f"mAP@0.5:0.95: {map50_95:.3f}")
print(f"결과 저장 위치: /content/drive/MyDrive/Final_project/runs/top_view_v2/")

if map50 >= 0.85:
    print("✅ 수락 기준 통과")
else:
    print("⚠️  수락 기준 미달 — confusion matrix 확인 후 추가 수집 검토")

In [ ]:
# confusion matrix 시각화
from IPython.display import Image

cm_path = "/content/drive/MyDrive/Final_project/runs/top_view_v2/confusion_matrix_normalized.png"
Image(cm_path)

In [ ]:
# 학습 곡선 시각화
Image("/content/drive/MyDrive/Final_project/runs/top_view_v2/results.png")